In [4]:
import numpy as np
import itertools
import json
import re
import chipsplitting as cs
from chipsplitting import pairing_matrix, PascalForm, LinearForm
from chipsplitting.hyperfield import HyperfieldVector as HV, HyperfieldHomogeneousLinearSystem as HLinSystem, grid_iter, HyperfieldLinearForm

In [5]:
CONTRACTION_SIZE = 5
DEGREE = 40
VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

In [6]:
def absolute(deg, c):
    if type(c) is int:
        return c

    if type(c) is str:
        if len(c) == 3:
            subtrahend = int(c[2])
            if subtrahend >= CONTRACTION_SIZE:
                raise Exception(f"Invalid subtrahend: {c}")
            return deg - int(c[2])
        elif c == 'd':
            return deg
        else:
            raise Exception(f'string {c} has not length 3')

    raise Exception(f"Invalid type. {c}")

def rel(degree, index):
    assert index < CONTRACTION_SIZE or index > degree - CONTRACTION_SIZE
    return f"d-{degree - index}" if index > CONTRACTION_SIZE else index

def parse_expression(expression):
    if expression[0] != '-':
        expression = '+' + expression
        
    ops = re.findall(r'[\+|-]',expression)
    summands = re.split(r'[\+|-]', expression[1:])
    return (summands, ops)

def realize_expression(degree, mode, op, unit):
    if op == '+':
        return PascalForm(degree, mode, unit)
    elif op == '-':
        return -PascalForm(degree, mode, unit)

"""
    Given an expression of pascal forms, find all units such that the expression is contractable.
"""
def find_contractables(degrees, expression):
    # d_begin = contraction_size * 3 - 1 + 3
    # d_end = contraction_size * 3 - 1 + 6
    # d_list = list(range(d_begin, d_end + 1))
    # if not short:
    #    d_list += [40, 41]

    res = []
    combinations = {}
    summands, ops = parse_expression(expression)

    for degree in degrees:
        units_domain = (
            list(range(CONTRACTION_SIZE)) + 
            list(range(degree - CONTRACTION_SIZE + 1, degree + 1))
        )
        for units in itertools.product(*([units_domain] * len(summands))):
            rel_units = tuple([rel(degree, u) for u in units])
            form = LinearForm.zero(degree)
            for mode, op, unit in zip(summands, ops, units):
                form = form + realize_expression(degree, mode, op, unit)

            if rel_units not in combinations:
                combinations[rel_units] = {}
                
            combinations[rel_units][degree] = {
                "contractable": form.is_contractable(CONTRACTION_SIZE), 
                "form": form
            }
        
    for rel_units, degrees_to_info in combinations.items():
        is_valid = True
        for info in degrees_to_info.values():
            if not info["contractable"]:
                is_valid = False
                break 
                
        if is_valid:
            res.append((rel_units, degrees_to_info))

    return res

In [7]:
%%time
expressions = [
    "col+row",
    "col-col",
    "col-diag",
    "row+row",
    "row-row",
    "row-col",
    "row-diag",
    "diag-diag",
    "diag+row+col",
    "row-diag+col",
    "col+row+col",
    "col+row-col",
    "col-row-col",
    "col+col-col",
    "diag-diag+col+row",
    "row+row+col+diag",
    "row+row+col-diag",
    "row+row-col-diag",
    "diag-diag+col+row+col",
    "diag-diag+col+row-col",
    "diag-diag+col-row-col",
    "diag-diag+diag+row+col",
    "diag-diag+diag+row-col",
    "diag-diag+col+col-col",
]

degrees = (DEGREE, DEGREE + 1)
d = degrees[0]

for expr in expressions:
    np.save(f"filter/{expr}", 
        [units for units, _ in find_contractables(degrees, expr)]
    )

CPU times: user 1h 5min 42s, sys: 6.25 s, total: 1h 5min 49s
Wall time: 1h 5min 49s


In [8]:
%%time
expressions = [
    "row+row+row+row",
    "row+row+row-row",
    "row+row-row-row",
    "row-row-row-row",
    "col+col+col+col",
    "col+col+col-col",
    "col+col-col-col",
    "col-col-col-col",
    "diag+diag+diag+diag",
    "diag+diag+diag-diag",
    "diag+diag-diag-diag",
    "diag-diag-diag-diag",
]

degrees = (DEGREE, DEGREE + 1)
d = degrees[0]

for expr in expressions:
    np.save(f"filter/{expr}", 
        [units for units, _ in find_contractables(degrees, expr)]
    )

CPU times: user 10min 43s, sys: 1.14 s, total: 10min 44s
Wall time: 10min 44s


In [9]:
%%time
expressions = [
    "col+col+col+col-col",
    "col+col+col-col-col",
    "col+col-col-col-col",
    "row+row+row+row-row",
    "row+row+row-row-row",
    "row+row-row-row-row",
]

degrees = (DEGREE, DEGREE + 1)
d = degrees[0]

for expr in expressions:
    np.save(f"filter/{expr}", 
        [units for units, _ in find_contractables(degrees, expr)]
    )

CPU times: user 1h 6min 26s, sys: 8.6 s, total: 1h 6min 34s
Wall time: 1h 6min 35s
